<a href="https://colab.research.google.com/github/onepartho/water-quality-analysis/blob/main/water_quality_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# River Water Quality Analysis: ML, 3D Manifold Projection & POV-Ray Export

**Source script:** `River_Water_Quality_ML_POVRay.py`

This master notebook converts the original Python pipeline into a Google Colab/Jupyter workflow while preserving its main analytical stages:

1. Data ingestion and audit
2. Gradient-boosted tree regression for WQI
3. Permutation feature importance
4. 3D t-SNE manifold projection
5. Export of 3D coordinates
6. POV-Ray scene generation
7. Diagnostic/publication figures

**Input workbook:** `Reproducible_Dataset.xlsx`  
**Expected sheet:** `Raw_Data`


## Important methodological note

The original script models **WQI from the same physicochemical variables used to construct WQI**. Therefore, the regression should be interpreted as a **reconstruction/association analysis**, not as independent validation of WQI or a prospective prediction model.

The original calculations are retained here so that the notebook reproduces the existing pipeline faithfully.


## 1. Import libraries and configure the analysis

In [1]:
import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.manifold import TSNE

RANDOM_STATE = 42

FEATURES = [
    'Alkalinity', 'BOD', 'COD', 'Chloride', 'DO',
    'EC', 'PH', 'SS', 'TDS', 'Turbidity'
]

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
POVRAY_DIR = OUTPUT_DIR / "povray_scenes"
PROCESSED_DIR = Path("data") / "processed"

for directory in [FIGURE_DIR, POVRAY_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Environment ready.")


Environment ready.


## 2. Load and audit the experimental dataset

In [2]:
candidate_paths = [
    Path("Reproducible_Dataset.xlsx"),
    Path("data/raw/Reproducible_Dataset.xlsx"),
    Path("../data/raw/Reproducible_Dataset.xlsx"),
    Path("../Reproducible_Dataset.xlsx"),
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Reproducible_Dataset.xlsx was not found. "
        "Upload it to the Colab session or place it in data/raw/."
    )

print(f"Loading data from: {dataset_path}")
df = pd.read_excel(dataset_path, sheet_name="Raw_Data")

X = df[FEATURES].copy()
y = df["WQI"].copy()
y_class = df["WQI_Class"].copy()

print(f"Dataset verified: {X.shape[0]} observations, {X.shape[1]} physical features.")
print(f"Missing values across features: {X.isnull().sum().sum()}")

display(df.head())
display(df[FEATURES + ["WQI", "WQI_Class"]].describe().T)


FileNotFoundError: Reproducible_Dataset.xlsx was not found. Upload it to the Colab session or place it in data/raw/.

## 3. Gradient-boosted tree regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Separate scaling for the unsupervised t-SNE analysis.
scaler_all = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X)

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.03,
    random_state=RANDOM_STATE
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test Set R² Score: {r2:.4f}")
print(f"Test Set MAE:      {mae:.4f} WQI units")
print(f"Test Set RMSE:     {rmse:.4f} WQI units")


## 4. Permutation feature importance

In [ ]:
perm = permutation_importance(
    model,
    X_test_scaled,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE
)

perm_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(perm_df)

print("\nPermutation Feature Importance Rankings (Drop in R²):")
for _, row in perm_df.iterrows():
    print(
        f"  {row['feature']:<12}: "
        f"{row['importance_mean']:.4f} +/- {row['importance_std']:.4f}"
    )


## 5. 3D t-SNE manifold projection

In [ ]:
print("Computing 3D non-linear manifold projection...")

manifold = TSNE(
    n_components=3,
    random_state=RANDOM_STATE,
    perplexity=30
)

coords_3d = manifold.fit_transform(X_scaled_all)

coords_3d -= coords_3d.mean(axis=0)
coords_3d /= np.abs(coords_3d).max()
coords_3d *= 5.0

df_coords = pd.DataFrame(coords_3d, columns=["X", "Y", "Z"])
df_coords["WQI"] = y.to_numpy()
df_coords["WQI_Class"] = y_class.to_numpy()
df_coords["River"] = df["River"].to_numpy()

coords_csv = PROCESSED_DIR / "manifold_embeddings_3d.csv"
df_coords.to_csv(coords_csv, index=False)

print(f"Saved 3D coordinates: {coords_csv}")
display(df_coords.head())


## 6. Generate the POV-Ray scene

In [ ]:
pov_path = POVRAY_DIR / "water_manifold.pov"

color_map_pov = {
    "Good": "rgb <0.18, 0.70, 0.25>",
    "Poor": "rgb <0.20, 0.50, 0.85>",
    "Very Poor": "rgb <1.00, 0.55, 0.10>",
    "Unsuitable for Drinking": "rgb <0.85, 0.15, 0.15>"
}

scene_header = '''// POV-Ray Ray-Traced 3D Manifold Scene
#version 3.7;
global_settings { assumed_gamma 1.0 }

camera {
    location <0, 8, -14>
    look_at <0, 0, 0>
    angle 45
}

light_source { <15, 20, -15> color rgb <1.0, 1.0, 1.0> }
light_source { <-15, 10, -10> color rgb <0.4, 0.4, 0.4> }
light_source { <0, -10, 5> color rgb <0.2, 0.2, 0.2> }

background { color rgb <0.96, 0.96, 0.98> }

box {
    <-5.5, -5.5, -5.5>, <5.5, 5.5, 5.5>
    pigment { color rgbt <0.75, 0.75, 0.80, 0.88> }
    finish { phong 0.1 }
}
'''

with open(pov_path, "w", encoding="utf-8") as f:
    f.write(scene_header)
    for (x_pt, y_pt, z_pt), cls in zip(coords_3d, y_class):
        pigment = color_map_pov.get(cls, "rgb <0.5, 0.5, 0.5>")
        f.write(
            f"sphere {{ <{x_pt:.4f}, {y_pt:.4f}, {z_pt:.4f}>, 0.18 "
            f"pigment {{ {pigment} }} "
            "finish { specular 0.6 roughness 0.02 reflection 0.04 } }\n"
        )

print(f"Generated POV-Ray scene file: {pov_path}")

povray_bin = shutil.which("povray")
if povray_bin:
    print(f"Found POV-Ray at '{povray_bin}'. Rendering photorealistic scene...")
    png_out = POVRAY_DIR / "water_manifold.png"
    subprocess.run([
        povray_bin, "+W1280", "+H720", "+A0.1", "+Q11",
        "+FN", str(pov_path), f"-O{png_out}"
    ], check=False)
    print(f"Rendered: {png_out}")
else:
    print("POV-Ray executable was not detected in the Colab/runtime PATH.")
    print(f"The scene file is ready for rendering in POV-Ray: {pov_path}")


## 7. Diagnostic figure: 3D manifold and WQI agreement

In [ ]:
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")

colors_mpl = {
    "Good": "#2ca02c",
    "Poor": "#1f77b4",
    "Very Poor": "#ff7f0e",
    "Unsuitable for Drinking": "#d62728"
}

for cls in ["Good", "Poor", "Very Poor", "Unsuitable for Drinking"]:
    mask = df_coords["WQI_Class"] == cls
    if mask.any():
        ax1.scatter(
            df_coords.loc[mask, "X"],
            df_coords.loc[mask, "Y"],
            df_coords.loc[mask, "Z"],
            c=colors_mpl[cls],
            label=cls,
            s=35,
            alpha=0.85,
            edgecolors="k",
            linewidth=0.3
        )

ax1.set_xlim([-5.5, 5.5])
ax1.set_ylim([-5.5, 5.5])
ax1.set_zlim([-5.5, 5.5])
ax1.set_xlabel("Manifold X", labelpad=10)
ax1.set_ylabel("Manifold Y", labelpad=10)
ax1.set_zlabel("Manifold Z", labelpad=10)
ax1.view_init(elev=25, azim=-60)
ax1.set_title(
    "3D Manifold Simulation (POV-Ray Camera Perspective)",
    fontsize=12,
    fontweight="bold"
)
ax1.legend(title="WQI Status", loc="upper left")

ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(
    y_test, y_pred,
    color="#2b5c8f",
    edgecolors="k",
    s=55,
    alpha=0.75,
    label="Holdout Test Observations"
)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

ax2.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--",
    lw=2,
    label="1:1 Ideal Fit Line"
)

ax2.set_xlabel("Actual Water Quality Index (WQI)", fontsize=11)
ax2.set_ylabel("Predicted WQI (Gradient Boosted Trees)", fontsize=11)
ax2.set_title(
    f"Regression Agreement (R² = {r2:.4f}, RMSE = {rmse:.2f})",
    fontsize=12,
    fontweight="bold"
)
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()

fig_out1 = FIGURE_DIR / "3d_manifold_and_validation.png"
plt.savefig(fig_out1, dpi=300, bbox_inches="tight")
print(f"Saved figure: {fig_out1}")

plt.show()


## 8. Feature importance and residual distribution

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=perm_df,
    x="importance_mean",
    y="feature",
    ax=axes[0],
    color="steelblue",
    edgecolor="k"
)

axes[0].errorbar(
    perm_df["importance_mean"],
    np.arange(len(perm_df)),
    xerr=perm_df["importance_std"],
    fmt="none",
    c="black",
    capsize=4
)

axes[0].set_title(
    "Permutation Feature Importance (Test Set)",
    fontsize=12,
    fontweight="bold"
)
axes[0].set_xlabel("Mean Drop in R²")
axes[0].set_ylabel("Parameter")
axes[0].grid(axis="x", linestyle=":", alpha=0.6)

residuals = y_test - y_pred

sns.histplot(
    residuals,
    kde=True,
    ax=axes[1],
    color="#1f77b4",
    edgecolor="k",
    alpha=0.7
)

axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_title(
    "Residual Error Distribution (Actual - Predicted)",
    fontsize=12,
    fontweight="bold"
)
axes[1].set_xlabel("Residual (WQI Units)")
axes[1].set_ylabel("Count")
axes[1].grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()

fig_out2 = FIGURE_DIR / "feature_importance_and_residuals.png"
plt.savefig(fig_out2, dpi=300, bbox_inches="tight")
print(f"Saved figure: {fig_out2}")

plt.show()


## 9. Export the main model metrics

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["R²", "MAE (WQI units)", "RMSE (WQI units)"],
    "Value": [r2, mae, rmse]
})

metrics_path = OUTPUT_DIR / "model_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df)
print(f"Saved: {metrics_path}")


## 10. Pipeline outputs

After successful execution, the notebook creates:

- `data/processed/manifold_embeddings_3d.csv`
- `outputs/figures/3d_manifold_and_validation.png`
- `outputs/figures/feature_importance_and_residuals.png`
- `outputs/model_metrics.csv`
- `outputs/povray_scenes/water_manifold.pov`
- `outputs/povray_scenes/water_manifold.png` *(only if POV-Ray is installed)*

The train/test split, model, permutation importance, and t-SNE use `random_state=42`, matching the original script.
